In [0]:
%sql
CREATE TABLE Emp (
    emp_id INT PRIMARY KEY,
    name VARCHAR(50),
    department_id INT,
    salary INT,
    manager_id INT
);


In [0]:
%sql
INSERT INTO Emp VALUES
(1, 'Alice', 101, 60000, NULL),
(2, 'Bob', 102, 45000, 1),
(3, 'Charlie', 101, 70000, 1),
(4, 'David', 103, 40000, 2),
(5, 'Eve', 102, 50000, 2);


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE TABLE Dept (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(50)
);


In [0]:
%sql
INSERT INTO Dept VALUES
(101, 'HR'),
(102, 'IT'),
(103, 'Finance');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
CREATE TABLE Order (
    order_id INT PRIMARY KEY,
    emp_id INT,
    order_amount INT,
    order_date DATE
);


In [0]:
%sql

INSERT INTO Order VALUES
(1, 1, 5000, '2024-01-10'),
(2, 2, 3000, '2024-02-15'),
(3, 3, 7000, '2024-03-12'),
(4, 1, 2000, '2024-04-01'),
(5, 4, 1000, '2024-05-05');


num_affected_rows,num_inserted_rows
5,5


1. Find employees who earn more than average salary

In [0]:
%sql
select * from Emp 
where salary > (select avg(salary) from Emp)

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1



2. Find employees who belong to the same department as 'Alice'


In [0]:
%sql
select emp_id 
from Emp
where department_id = (
  select department_id from Emp
  where name='Alice'
)

emp_id
1
3


3. Get employees whose salary is equal to the minimum salary


In [0]:
%sql
select emp_id 
from Emp 
where salary = (select min(salary) from Emp)

emp_id
4


Find employees who have placed at least one order


In [0]:
%sql
select * from Emp e 
where exists (
  select 1 
  from order o
  where o.emp_id=e.emp_id
)


emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
2,Bob,102,45000,1
3,Charlie,101,70000,1
4,David,103,40000,2


5. Get employees whose salary is greater than ALL employees in department 102


In [0]:
%sql
select emp_id from Emp 
where salary > (
  select MAX(salary) from Emp
  where department_id = 102
)

emp_id
1
3


CTE Questions

Create a CTE to show employees with salary > 50,000 and fetch all records


In [0]:
%sql
with high_salary_emp as(
  select * from Emp 
  where salary > 50000
)
select * from high_salary_emp


emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


2. Find average salary per department using CTE


In [0]:
%sql
with avg_salary_dept as(
    select avg(salary) from Emp 
    group by department_id
)
select * from avg_salary_dept

avg(salary)
65000.0
47500.0
40000.0


3. Use CTE to get employees and their department names


In [0]:
%sql
with emp_with_dept as(
  select emp_id, name, department_id from Emp
)
select e.emp_id,e.name, d.dept_id, d.dept_name 
from emp_with_dept e
join Dept d
on e.department_id=d.dept_id

emp_id,name,dept_id,dept_name
1,Alice,101,HR
2,Bob,102,IT
3,Charlie,101,HR
4,David,103,Finance
5,Eve,102,IT


4. Find total order amount per employee using CTE

In [0]:
%sql
with total_amount_employee as(
    select emp_id, sum(order_amount) as total_order
    from order
    group by emp_id
)select * from total_amount_employee

emp_id,total_order
1,7000
2,3000
3,7000
4,1000


5. Find employees whose salary is above department average using CTE


In [0]:
%sql
with avg_salary as(
    select department_id, avg(salary) as avg_salary
    from Emp 
    group by department_id
)
select e.emp_id, e.name, e.salary, a.avg_salary
from Emp e
join avg_salary a on e.department_id = a.department_id
where e.salary > a.avg_salary

emp_id,name,salary,avg_salary
3,Charlie,70000,65000.0
5,Eve,50000,47500.0
